In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
document = PyPDFLoader('ml_paper1.pdf')
text_content = document.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
content = text_splitter.split_documents(text_content)
content[1].page_content

'various models and put forward a detailed understanding of the past, present and future of deep learning in NLP.\nIndex Terms\nNatural Language Processing, Deep Learning, Word2Vec, Attention, Recurrent Neural Networks, Convolutional Neural Net-\nworks, LSTM, Sentiment Analysis, Question Answering, Dialogue Systems, Parsing, Named-Entity Recognition, POS Tagging,\nSemantic Role Labeling\nI. I NTRODUCTION\nNatural language processing (NLP) is a theory-motivated range of computational techniques for the automatic analysis and\nrepresentation of human language. NLP research has evolved from the era of punch cards and batch processing, in which the\nanalysis of a sentence could take up to 7 minutes, to the era of Google and the likes of it, in which millions of webpages can\nbe processed in less than a second [1]. NLP enables computers to perform a wide range of natural language related tasks at'

In [13]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from dotenv import load_dotenv
import os
load_dotenv()
api_key=os.getenv('GEMINI_API_KEY')
embeddings = GoogleGenerativeAIEmbeddings(
    model='gemini-embedding-001',
    google_api_key=api_key
)
db = FAISS.from_documents(content[:5], embeddings)

In [14]:
query = 'What is natural language processing?'
result = db.similarity_search(query)
result[1].page_content

'all levels, ranging from parsing and part-of-speech (POS) tagging, to machine translation and dialogue systems.\nDeep learning architectures and algorithms have already made impressive advances in ﬁelds such as computer vision and\npattern recognition. Following this trend, recent NLP research is now increasingly focusing on the use of new deep learning\nmethods (see Figure 1). For decades, machine learning approaches targeting NLP problems have been based on shallow models\n(e.g., SVM and logistic regression) trained on very high dimensional and sparse features. In the last few years, neural networks\nbased on dense vector representations have been producing superior results on various NLP tasks. This trend is sparked by\nthe success of word embeddings [2, 3] and deep learning methods [4]. Deep learning enables multi-level automatic feature\nrepresentation learning. In contrast, traditional machine learning based NLP systems liaise heavily on hand-crafted features.'

In [15]:
from langchain_ollama import OllamaLLM
llm = OllamaLLM(model='gemma3:1b')

In [16]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template("""
Answer the following questions based on the provided context only. Think step-by-step before providing any
                                          answer. 1000 reward points will be awarded to you for every 
                                          correct answer.
                                          <context>
                                          {context}
                                          </context>
                                          Question:{input}
""")

In [17]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
document_chain = create_stuff_documents_chain(llm, prompt)

In [18]:
retriever = db.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001BDC69AD040>, search_kwargs={})

In [20]:
from langchain_classic.chains import create_retrieval_chain
retrieval_chain = create_retrieval_chain(retriever, document_chain)
response=retrieval_chain.invoke({"input": "Natural language processing (NLP) is a theory-motivated range of computational technique"})

In [21]:
response['answer']

"Okay, let's answer the questions based on the provided context.\n\n**1. What is the main focus of the text?**\n\nThe text focuses on the evolution of Deep Learning in Natural Language Processing (NLP). It covers the historical development of NLP, the introduction of deep learning, and the various models and methods that have been developed since.\n\n**2. What are some of the key developments mentioned in the text?**\n\n*   **Historical Context:** The text begins by providing a historical overview of NLP, tracing its evolution from punch cards to Google’s use of millions of webpages.\n*   **Deep Learning Introduction:** It explains how deep learning has advanced in areas like computer vision and pattern recognition and then transitioned to focusing on new deep learning methods.\n*   **Word Embeddings:** The text highlights the success of word embeddings as a foundation for deep learning.\n*   **Model Review:** It outlines various deep learning models and techniques, including CNNs, RNN